# Threshold Selection Analysis



In [116]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from itertools import cycle

## Config

In [ ]:
SAVE = False
# ── ────────────────────────────────────────────────────────────────────
# CSV columns: m, n, t, model, gtruth, retrieved, precision, recall,
#              tp, fp, fn, tn, repetition, qid, query
DATASET_NAME = "movie"   # appears in plot titles
TASK = "threshold"
SELECT_TYPE = "nonequality"
DATA_PATH = f"../data/processed/{TASK}_{SELECT_TYPE}_{DATASET_NAME}.csv"

# ────────────────────────────────────────────────────────────────────
# Options: 'f1'  |  'precision'  |  'recall'
METRIC = 'f1'

# ───────────────────────────────────────────────────────────
SHOW_TAU_LINE = True    # show the sqrt(n/m) reference line
TAU_LABEL     = False    # annotate the tau line with its value
SHOW_GRID     = True
AXIS_MARGIN   = 0.02    # padding around [0, 1] on both axes
FIG_SIZE      = (14, 10)  # overall grid figure size
ZOOM_FIG_SIZE = (7, 5)    # figure size for the zoomed single-cell plot

# X-axis ticks can be changed in one place for both the grid and zoomed plot.
if SELECT_TYPE == "equality" and TASK == "threshold":
    XLIMITS = (0.1 - AXIS_MARGIN, 1 + AXIS_MARGIN)
    YLIMITS = (0 - AXIS_MARGIN, 1 + AXIS_MARGIN)

    GRID_XTICKS = [0.1, 1] 
    GRID_XTICK_LABELS = ['0.1', '1.0']   
    GRID_YTICKS = [0.0, 0.5, 1.0]
    GRID_YTICK_LABELS = ['0.0', '0.5', '1.0']   

    QUADRANT_XTICKS = [0.1, 1.0]
    QUADRANT_XTICK_LABELS = ['0.1', '1.0']
    QUADRANT_YTICKS = GRID_YTICKS
    QUADRANT_YTICK_LABELS = GRID_YTICK_LABELS

elif SELECT_TYPE == "nonequality":
    XLIMITS = (-0.3 - AXIS_MARGIN, 0.2 + AXIS_MARGIN)
    YLIMITS = (0 - AXIS_MARGIN, 1 + AXIS_MARGIN)

    GRID_XTICKS = [-0.3, 0.2] 
    GRID_XTICK_LABELS = ['-0.3', '0.2']   
    GRID_YTICKS = [0.0, 0.5, 1.0]
    GRID_YTICK_LABELS = ['0.0', '0.5', '1.0']   

    QUADRANT_XTICKS = [-0.3, 0.2]
    QUADRANT_XTICK_LABELS = ['-0.3', '0.2']
    QUADRANT_YTICKS = GRID_YTICKS
    QUADRANT_YTICK_LABELS = GRID_YTICK_LABELS

elif SELECT_TYPE == "equality" and TASK == "topk":
    XLIMITS = (1 - AXIS_MARGIN, 20 + AXIS_MARGIN)
    YLIMITS = (0 - AXIS_MARGIN, 1 + AXIS_MARGIN)

    GRID_XTICKS = [1, 10, 20] 
    GRID_XTICK_LABELS = ['1', '10', '20']   
    GRID_YTICKS = [0.0, 0.5, 1.0]
    GRID_YTICK_LABELS = ['0.0', '0.5', '1.0']   

    QUADRANT_XTICKS = GRID_XTICKS
    QUADRANT_XTICK_LABELS = GRID_XTICK_LABELS
    QUADRANT_YTICKS = GRID_YTICKS
    QUADRANT_YTICK_LABELS = GRID_YTICK_LABELS

FONT_SIZE = 12

# ────────────────────────────────────────────────────
# Models in the same family share a hue; variants differ in lightness & linestyle.
# Members listed explicitly — no magic prefix matching needed.
METHOD_FAMILIES = {
    'HRR': [
        'HRR_300', 'HRR_512', 'HRR_1024', #'HRR_2048',
    ],
    # 'BSC': [
    #     'BSC_300', 'BSC_512', 'BSC_1024', 'BSC_2048', 'BSC_4096', 'BSC_8192',
    # ],
    # 'SemHDC': [
    #     'SemHDC_FastText_300',
    # ],
    'EmbDI': [
        'EmbDI_300', 'EmbDI_512', 
    ],
}

# Base hues per family (degrees, 0–360).  Add a key here if you add a new family.
FAMILY_HUES = {
    'HRR':    210,   # blues
    'BSC':    30,    # oranges
    'SemHDC': 140,   # greens
    'EmbDI':  290,   # purples
    '_other': 0,     # gray fallback
}

## Load & prepare data

In [118]:
# CSV columns: m, n, t, model, gtruth, retrieved, precision, recall,
#              tp, fp, fn, tn, repetition, qid, query
df = pd.read_csv(DATA_PATH)
if SELECT_TYPE == "nonequality":
    df = df[df['gtruth'] > 0] # filter out zero-ground-truth queries (undefined precision/recall)

# if TASK == "topk":
#     df = df[(df['n'].isin([1, 5, 10, 15])) & (df['m'].isin([1, 5, 10, 15]))]

# Cast numeric columns
for col in ['t' if TASK == "threshold" else 'topk', 'recall', 'precision', 'tp', 'fp', 'fn']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Compute F1 from precision/recall (falls back to tp/fp/fn if needed)
if 'f1' not in df.columns:
    denom = df['recall'] + df['precision']
    df['f1'] = np.where(
        denom > 0,
        2 * df['recall'] * df['precision'] / denom,
        0.0
    )

# Aggregate over repetitions / queries
agg_df = (
    df.groupby(['model', 'n', 'm', 't' if TASK == "threshold" else 'topk'], as_index=False)
      .agg(recall=('recall', 'mean'),
           precision=('precision', 'mean'),
           f1=('f1', 'mean'))
)

q_lengths  = np.sort(agg_df['n'].unique())   # grid rows  (query length)
c_nums     = np.sort(agg_df['m'].unique())   # grid cols  (# columns)
all_models = sorted(agg_df['model'].unique())

print(f"Rows (n) : {list(q_lengths)}")
print(f"Cols (m) : {list(c_nums)}")
print(f"Models   : {all_models}")
print(f"Metric   : {METRIC}")

Rows (n) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Cols (m) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Models   : ['EmbDI_300', 'EmbDI_512', 'HRR_1024', 'HRR_300', 'HRR_512']
Metric   : f1


## Build color palette

In [119]:
import colorsys
import re

def hsl_to_rgb(h, s, l):
    """h in [0, 360], s and l in [0, 1]."""
    return colorsys.hls_to_rgb(h / 360, l, s)

def model_dimension_key(model):
    """Sort models by numeric dimension when present; fall back to name."""
    match = re.search(r'(\d+)(?!.*\d)', model)
    return (int(match.group(1)) if match else float('inf'), model)

def make_family_colors(models, family_map, hue_map):
    """Return (palette, linestyles, markers, assigned) dicts keyed by model name.

    Each family shares one base hue (from `hue_map`). Members within a family
    share that hue but receive distinct lightness (intensity), line styles and
    markers so they are visually distinguishable while remaining color-consistent.

    Assignment priority:
      1. Explicit member list in family_map.
      2. Prefix match (family name is a prefix of the model name, case-insensitive).
      3. '_other' → gray.
    """
    from collections import defaultdict

    # Build reverse lookup: member → family
    explicit = {}
    for fam, members in family_map.items():
        for m in members:
            explicit[m] = fam

    assigned = {}
    for model in models:
        if model in explicit:
            assigned[model] = explicit[model]
        else:
            # prefix fallback
            matched = False
            for fam in family_map:
                if model.upper().startswith(fam.upper()):
                    assigned[model] = fam
                    matched = True
                    break
            if not matched:
                assigned[model] = '_other'

    family_members = defaultdict(list)
    for model, fam in assigned.items():
        family_members[fam].append(model)

    palette    = {}
    linestyles = {}
    markers    = {}

    # Line style, marker and intensity cycles to differentiate family members
    style_cycle = ['-', (0, (0.7, 0.3)), '-.', ':']
    marker_cycle = ['', '', 's', '^', 'D']
    # Lightness/intensity values (lower -> darker). Will be cycled per-member.
    intensity_cycle = [0.40, 0.6, 0.46, 0.54, 0.62]

    for fam, members in family_members.items():
        hue = hue_map.get(fam, 0)
        members = sorted(members, key=model_dimension_key)
        n   = len(members)

        if fam == '_other':
            # Grayscale variants for unknown families
            grays = np.linspace(0.40, 0.70, max(n, 1))
            for k, model in enumerate(members):
                g = grays[k]
                palette[model] = (g, g, g)
                linestyles[model] = style_cycle[k % len(style_cycle)]
                markers[model] = marker_cycle[k % len(marker_cycle)]
        else:
            # Base saturation kept constant; vary lightness per member using intensity_cycle
            sat = 0.75
            for k, model in enumerate(members):
                lightness = intensity_cycle[k % len(intensity_cycle)]
                palette[model] = hsl_to_rgb(hue, sat, lightness)
                linestyles[model] = style_cycle[k % len(style_cycle)]
                markers[model] = marker_cycle[k % len(marker_cycle)]

    return palette, linestyles, markers, assigned

COLOR_MAP, LS_MAP, MK_MAP, FAMILY_ASSIGNED = make_family_colors(
    all_models, METHOD_FAMILIES, FAMILY_HUES
)

print("Color assignments (family → models):")
from collections import defaultdict
_by_fam = defaultdict(list)
for mdl, fam in FAMILY_ASSIGNED.items():
    _by_fam[fam].append(mdl)
for fam, mdls in sorted(_by_fam.items()):
    ordered = sorted(mdls, key=model_dimension_key)
    print(f"  {fam:10s}: {', '.join(sorted(mdls))}")


Color assignments (family → models):
  EmbDI     : EmbDI_300, EmbDI_512
  HRR       : HRR_1024, HRR_300, HRR_512


## Shared plotting helper

In [120]:
METRIC_LABEL = {'f1': 'F1 Score', 'precision': 'Precision', 'recall': 'Recall'}

THRESHOLD_COLORS = [
    hsl_to_rgb(30, 0.75, 0.5),
    hsl_to_rgb(150, 0.75, 0.3),
    hsl_to_rgb(140, 0.75, 0.4)
]

THRESHOLD_LINES = [':', '-', '-']
THRESHOLD_LINES = [(0, (0.6, 0.5)), (0, (0.9, 0.25)), (0, (0.8, 0.6))]


def plot_one(ax, fdf, metric, show_tau=True, show_legend=False,
             xlimits=XLIMITS, ylimits=YLIMITS, show_grid=SHOW_GRID,
             line_width=1.8, marker_size=5, tau_linewidth=1.2,
             legend_fontsize=8, xticks=None, xtick_labels=None,
             yticks=None, ytick_labels=None):
    """Draw one subplot on *ax* using the pre-built global palette.

    Families are drawn in a specific order (EmbDI first, then HRR); within each
    family higher-dimension models are drawn first so lower-dimension variants
    appear on top.
    """
    if fdf.empty:
        return

    # Desired family order: EmbDI first, then HRR; append remaining families
    preferred = ['EmbDI', 'HRR']
    remaining = [f for f in list(METHOD_FAMILIES.keys()) if f not in preferred]
    ordered_families = preferred + remaining + ['_other']

    present_models = list(fdf['model'].unique())

    for fam in ordered_families:
        # models in this family that are present in the dataframe
        members = [m for m in present_models if FAMILY_ASSIGNED.get(m, '_other') == fam]
        if not members:
            continue
        # sort so higher-dimension (larger numeric suffix) plotted first
        members = sorted(members, key=model_dimension_key, reverse=True)
        for model in members:
            grp = fdf[fdf['model'] == model]
            if grp.empty:
                continue
            color = COLOR_MAP.get(model, (0.5, 0.5, 0.5))
            ls    = LS_MAP.get(model, '-')
            mk    = MK_MAP.get(model, '')
            ax.plot(grp['t' if TASK == "threshold" else 'topk'], grp[metric],
                    label=model, color=color,
                    linestyle=ls, marker=mk,
                    markersize=marker_size, linewidth=line_width)

    if show_tau and len(fdf) > 0:
        n_val = fdf['n'].iloc[0]
        m_val = fdf['m'].iloc[0]
        if SELECT_TYPE == "equality":
            tau = (n_val / m_val) ** 0.5
            # sigma_300 = ((4*m_val + 2*n_val + 9)/(2*m_val*300)) ** 0.5
            # sigma_512 = ((2*m_val + 1*n_val + 5)/(1*m_val*300)) ** 0.5
            # sigma_512 = ((4*m_val + 2*n_val + 9)/(2*m_val*512)) ** 0.5

            # sigma_alt = ((2*m_val-2+n_val)/(m_val*300)) ** 0.5

            sigma_300 = ((8*300*m_val - 8*300 + 8*m_val - 8 + 4*300*n_val + n_val)/(4*300*300*m_val)) ** 0.5
            sigma_512 = ((8*512*m_val - 8*512 + 8*m_val - 8 + 4*512*n_val + n_val)/(4*512*512*m_val)) ** 0.5
            # print(tau-sigma_300, tau-sigma_512, tau-sigma_alt)
        else:
            tau = 0
            sigma_300 = ((2 * 300 + 2) / (300 * 300)) ** 0.5
            sigma_512 = ((2 * 512 + 2) / (512 * 512)) ** 0.5
        
        ax.axvline(x=tau-sigma_512, color=THRESHOLD_COLORS[2], linestyle=THRESHOLD_LINES[2], 
                   linewidth=tau_linewidth, label=f'τ512={tau-sigma_512:.2f}')
        
        ax.axvline(x=tau-sigma_300, color=THRESHOLD_COLORS[1], linestyle=THRESHOLD_LINES[1], 
                   linewidth=tau_linewidth, label=f'τ300={tau-sigma_300:.2f}')
        
        ax.axvline(x=tau, color=THRESHOLD_COLORS[0], linestyle=THRESHOLD_LINES[0], 
                   linewidth=tau_linewidth, label=f'τ={tau:.2f}')
        

    if xlimits is None and len(fdf) > 0:
        xlimits = (float(fdf['t' if TASK == "threshold" else 'topk'].min()), float(fdf['t' if TASK == "threshold" else 'topk'].max()))
    if xlimits is None:
        xlimits = XLIMITS
    ax.set_xlim(*xlimits)
    ax.set_ylim(*ylimits)
    if show_grid:
        ax.grid(True, linestyle='--', alpha=0.4)

    if xticks is not None:
        ax.set_xticks(xticks)
        if xtick_labels is not None:
            ax.set_xticklabels(xtick_labels, fontsize=legend_fontsize)
    if yticks is not None:
        ax.set_yticks(yticks)
        if ytick_labels is not None:
            ax.set_yticklabels(ytick_labels, fontsize=legend_fontsize)

    if show_legend:
        ax.legend(fontsize=legend_fontsize, loc='lower right', framealpha=0.8)


def make_legend_handles(models, show_tau=True, linewidth=2.2):
    """Global legend: one entry per model, grouped by family, with family headers."""
    from collections import defaultdict
    by_family = defaultdict(list)
    for model in models:
        by_family[FAMILY_ASSIGNED.get(model, '_other')].append(model)

    handles = []
    # Respect the order defined in METHOD_FAMILIES, then '_other'
    ordered_fams = list(METHOD_FAMILIES.keys()) + ['_other']
    for fam in ordered_fams:
        members = sorted(by_family.get(fam, []), key=model_dimension_key)
        if not members:
            continue
        for model in members:
            handles.append(Line2D([0], [0],
                                  color=COLOR_MAP[model],
                                  linestyle=LS_MAP[model],
                                  marker=MK_MAP.get(model, ''),
                                  markersize=6,
                                  linewidth=linewidth,
                                  label=f'{model.replace("_", " ")}'))
        
        # Family header (invisible line, bold label)
        handles.append(Line2D([0], [0], color='none',
                              label=''))
                            #    label='f'── {fam} ──')')
    # τ reference line
    if show_tau:
        handles.append(Line2D([0], [0], color=THRESHOLD_COLORS[1], linestyle=THRESHOLD_LINES[1], linewidth=linewidth,
                               label='τ 300'))
        handles.append(Line2D([0], [0], color=THRESHOLD_COLORS[2], linestyle=THRESHOLD_LINES[2], linewidth=linewidth,
                           label='τ 512'))
        handles.append(Line2D([0], [0], color=THRESHOLD_COLORS[0], linestyle=THRESHOLD_LINES[0], linewidth=linewidth,
                               label='√(n/m)' if SELECT_TYPE == "equality" else 'τ = 0'))
    return handles


## Full grid plot

In [ ]:
MODELS_IN = [
    "EmbDI_300",
    "EmbDI_512",
    # "BSC_300", "BSC_512", "BSC_1024", "BSC_2048", "BSC_4096", "BSC_8192",
    "HRR_300", 
    "HRR_512", 
    # "HRR_1024", 
    # "HRR_2048",
    # "SemHDC_FastText_300",
]

nrows, ncols = len(q_lengths), len(c_nums)

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=FIG_SIZE,
    sharex=True, sharey=True,
    constrained_layout=False
)
plt.subplots_adjust(left=0.07, right=0.80, top=0.90, bottom=0.08,
                    hspace=0.25, wspace=0.15)

# Ensure axes is always 2-D
if nrows == 1: axes = axes[np.newaxis, :]
if ncols == 1: axes = axes[:, np.newaxis]

for i, q_len in enumerate(q_lengths):
    for j, c_num in enumerate(c_nums):
        ax = axes[i, j]
        if j < i:                       # hide lower-triangle (optional)
            ax.set_visible(False)
            continue
        fdf = agg_df[(agg_df['n'] == q_len) & (agg_df['m'] == c_num) & (agg_df['model'].isin(MODELS_IN))]
        plot_one(ax, fdf, METRIC, show_tau=False, legend_fontsize=FONT_SIZE,
                 xticks=GRID_XTICKS, xtick_labels=GRID_XTICK_LABELS,
                 yticks=GRID_YTICKS, ytick_labels=GRID_YTICK_LABELS)

# Column headers (top) = m values
for j, c_num in enumerate(c_nums):
    axes[0, j].set_title(f'm = {int(c_num)}', fontsize=FONT_SIZE, pad=4)

# Row labels (right side of grid) = n values
for i, q_len in enumerate(q_lengths):
    axes[i, -1].yaxis.set_label_position('right')
    axes[i, -1].set_ylabel(f'n = {int(q_len)}', rotation=0,
                            labelpad=23, fontsize=FONT_SIZE, va='center')

# Axis labels on border cells
for ax in axes[-1, :]:
    ax.set_xlabel('Threshold' if TASK == "threshold" else 'Top-k', fontsize=FONT_SIZE)
for ax in axes[:, 0]:
    ax.set_ylabel(METRIC_LABEL[METRIC], fontsize=FONT_SIZE)

# Legend in the right margin (stays readable with many models)
fig.legend(handles=make_legend_handles(MODELS_IN, show_tau=False),
           loc='center left',
           bbox_to_anchor=(0.1, 0.25),
           fontsize=FONT_SIZE, framealpha=0.9,
           handlelength=2.4)

# fig.suptitle(
#     f"{METRIC_LABEL[METRIC]} — {DATASET_NAME} dataset\n"
#     f"rows = query length (n),  cols = # table columns (m)",
#     fontsize=13
# )

if SAVE:
    plt.savefig(f"../outputs/plots/{SELECT_TYPE}_{DATASET_NAME}_grid.png", dpi=300)
else:
    plt.show()

## specific (n, m) cell in detail

Change `ZOOM_N` and `ZOOM_M` to any values from the grid, then run the cell.

In [122]:
def plot_quadrants(models, m, n, ylabel=True, xlabel=True, title=True, show_legend=True, save=True,
                   xlimits=XLIMITS, ylimits=YLIMITS, line_width=8, marker_size=5, tau_linewidth=6,
                   title_fontsize=20, label_fontsize=30, legend_fontsize=25):
    fdf = agg_df[(agg_df['n'] == n) & (agg_df['m'] == m) & (agg_df['model'].isin(models))]

    if fdf.empty:
        print(f"No data for n={n}, m={m}. "
            f"Available n: {list(q_lengths)}, m: {list(c_nums)}")
    else:
        fig, ax = plt.subplots(figsize=(10, 8))
        plot_one(ax, fdf, METRIC, show_tau=SHOW_TAU_LINE, show_legend=False,
                 xlimits=xlimits, ylimits=ylimits, line_width=line_width,
                 marker_size=marker_size, legend_fontsize=label_fontsize,
                 tau_linewidth=tau_linewidth,
                 xticks=QUADRANT_XTICKS, xtick_labels=QUADRANT_XTICK_LABELS,
                 yticks=QUADRANT_YTICKS, ytick_labels=QUADRANT_YTICK_LABELS)
        # for line in ax.get_lines():
        #     line.set_linewidth(line_width) 
        if show_legend:
            legend = ax.legend(handles=make_legend_handles(models, linewidth=line_width + 1),
                               loc='upper right', fontsize=legend_fontsize, framealpha=0.9)

        ax.set_xlabel('Threshold', fontsize=label_fontsize, color='black' if xlabel else 'white')
        ax.set_ylabel(METRIC_LABEL[METRIC], fontsize=label_fontsize, color='black' if ylabel else 'white')
        ax.set_title(
            f"{METRIC_LABEL[METRIC]} — {DATASET_NAME}\n"
            f"query length n={n},  columns m={m}",
            fontsize=title_fontsize,
            color='black' if title else 'white'
        )
        ax.set_box_aspect(8 / 10)

        # plt.tight_layout()
        # plt.savefig(
        #     f"../outputs/figures/equality_movie_m{m}_n{n}_legend.png",
        #     bbox_inches='tight',
        # )
        # legend.remove()
        if save:
            plt.savefig(
                f"../outputs/plots/{SELECT_TYPE}_{DATASET_NAME}_m{m}_n{n}.png",
                bbox_inches='tight',
            )
            plt.close()
        else:
            plt.show()


plot_quadrants(MODELS_IN, m=15, n=1, title=False, ylabel=True, show_legend=False, save=SAVE)
plot_quadrants(MODELS_IN, m=15, n=3, title=False, ylabel=False, show_legend=False, save=SAVE)
plot_quadrants(MODELS_IN, m=15, n=7, title=False, ylabel=False, show_legend=False, save=SAVE)
plot_quadrants(MODELS_IN, m=15, n=15, title=False, ylabel=False, show_legend=True, save=SAVE)